# MedCLIP-SAMv2 Thigh Segmentation — Augmented Dataset (Lambda)

Runs [MedCLIP-SAMv2](https://github.com/HealthX-Lab/MedCLIP-SAMv2) zero-shot segmentation
on the augmented dataset water Dixon MRI stacks.

Pipeline per stack per muscle:
1. Export NIfTI slices to PNG
2. BiomedCLIP saliency map (text prompt → heatmap per slice)
3. Postprocessing (kmeans → coarse binary mask)
4. SAM refinement (coarse mask → precise boundary)
5. Reassemble PNG masks → 3D NPZ

Data: `~/our_augmented_dataset/{stem}_augmented000_water.nii.gz`
Output: `~/medclipsamv2_augmented_segs/{stem}_medclipsamv2.npz`

## 1 — Upload to Lambda
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /path/to/our_augmented_dataset/ \
  ubuntu@<YOUR-LAMBDA-IP>:~/our_augmented_dataset/
```

## 2 — Download results when done
```bash
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<YOUR-LAMBDA-IP>:~/medclipsamv2_augmented_segs/ \
  /path/to/local/medclipsamv2/augmented_segs/
```
**Terminate the instance when done.**

In [ ]:
import subprocess, sys, os

REPO_DIR = os.path.expanduser('~/MedCLIP-SAMv2')
VENV_DIR = os.path.expanduser('~/mcsam2_env')
VENV_PY  = os.path.join(VENV_DIR, 'bin', 'python')

# ── Clone repo ───────────────────────────────────────────────────────────────
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone',
        'https://github.com/HealthX-Lab/MedCLIP-SAMv2.git', REPO_DIR])
    print('Cloned MedCLIP-SAMv2')
else:
    print('Repo already present')

# ── Create a clean venv (no --system-site-packages) ─────────────────────────
if not os.path.exists(VENV_PY):
    subprocess.check_call([sys.executable, '-m', 'venv', VENV_DIR])
    print('Venv created:', VENV_DIR)
else:
    print('Venv already exists:', VENV_DIR)

def venv_pip(*args):
    subprocess.check_call([VENV_PY, '-m', 'pip'] + list(args))

venv_pip('install', '-q', '--upgrade', 'pip')
venv_pip('install', '-q', 'numpy', 'scikit-learn')
venv_pip('install', '-q', '--upgrade',
    'torch', 'torchvision',
    '--index-url', 'https://download.pytorch.org/whl/cu124')
venv_pip('install', '-q', '-e', os.path.join(REPO_DIR, 'segment-anything'))
venv_pip('install', '-q',
    'git+https://github.com/lucasb-eyer/pydensecrf.git')
venv_pip('install', '-q',
    'open_clip_torch',
    'opencv-python',
    'SimpleITK',
    'Pillow',
    'huggingface_hub',
    'transformers<4.46',
    'matplotlib',
    'grad-cam',
    'pandas',
    'tqdm',
    'scipy',
)
print('All dependencies installed.')

In [ ]:
import os, urllib.request

CKPT_DIR  = os.path.join(os.path.expanduser('~/MedCLIP-SAMv2'),
                          'segment-anything', 'sam_checkpoints')
CKPT_FILE = os.path.join(CKPT_DIR, 'sam_vit_b_01ec64.pth')
URL = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'

os.makedirs(CKPT_DIR, exist_ok=True)
if not os.path.exists(CKPT_FILE):
    print('Downloading SAM ViT-B checkpoint (~375 MB)...')
    urllib.request.urlretrieve(URL, CKPT_FILE)
    print(f'Done ({os.path.getsize(CKPT_FILE) // 1_000_000} MB)')
else:
    print('SAM checkpoint already present')

In [ ]:
import glob, shutil, tempfile
import numpy as np
import SimpleITK as sitk
import torch
import cv2
from PIL import Image

DATA_DIR   = os.path.expanduser('~/our_augmented_dataset')
REPO_DIR   = os.path.expanduser('~/MedCLIP-SAMv2')
VENV_PY    = os.path.expanduser('~/mcsam2_env/bin/python')
VENV_DIR   = os.path.expanduser('~/mcsam2_env')
SAM_CKPT   = os.path.join(REPO_DIR, 'segment-anything', 'sam_checkpoints', 'sam_vit_b_01ec64.pth')
SAM_TYPE   = 'vit_b'
OUTPUT_DIR = os.path.expanduser('~/medclipsamv2_augmented_segs')

os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device  :', DEVICE)
print('VENV_PY :', VENV_PY, '— exists:', os.path.exists(VENV_PY))
print('SAM ckpt:', os.path.exists(SAM_CKPT))

nii_files = sorted(glob.glob(os.path.join(DATA_DIR, '*_augmented*_water.nii.gz')))
print(f'Found : {len(nii_files)} water NIfTI volumes')

In [ ]:
# Muscle definitions: (output_key, text_prompt)
MUSCLES = [
    ('R_gracilis',
     'gracilis muscle right thigh Dixon MRI axial cross section'),
    ('L_gracilis',
     'gracilis muscle left thigh Dixon MRI axial cross section'),
    ('R_sartorius',
     'sartorius muscle right thigh Dixon MRI axial cross section'),
    ('L_sartorius',
     'sartorius muscle left thigh Dixon MRI axial cross section'),
]

SAL_SCRIPT  = os.path.join(REPO_DIR, 'saliency_maps', 'generate_saliency_maps.py')
POST_SCRIPT = os.path.join(REPO_DIR, 'postprocessing', 'postprocess_saliency_maps.py')
SAM_SCRIPT  = os.path.join(REPO_DIR, 'segment-anything', 'prompt_sam.py')

for s in [SAL_SCRIPT, POST_SCRIPT, SAM_SCRIPT]:
    print('exists:', os.path.exists(s), s)

In [ ]:
def export_slices_as_png(img_array, out_dir):
    """Write (D, H, W) float array as 0-indexed RGB PNGs for MedCLIP-SAMv2."""
    os.makedirs(out_dir, exist_ok=True)
    for i in range(img_array.shape[0]):
        sl = img_array[i]
        sl_norm  = (sl - sl.min()) / (sl.max() - sl.min() + 1e-8)
        sl_uint8 = (sl_norm * 255).astype(np.uint8)
        rgb = np.stack([sl_uint8] * 3, axis=-1)
        Image.fromarray(rgb).save(os.path.join(out_dir, f'{i}.png'))


import glob as _glob
_venv_site = _glob.glob(os.path.join(VENV_DIR, 'lib', 'python3.*', 'site-packages'))
if not _venv_site:
    raise RuntimeError(f'Could not find site-packages inside {VENV_DIR}')
VENV_SITE = _venv_site[0]
print('Venv site-packages:', VENV_SITE)

SUBPROCESS_ENV = os.environ.copy()
SUBPROCESS_ENV['PYTHONPATH'] = VENV_SITE + ':' + os.environ.get('PYTHONPATH', '')
SUBPROCESS_ENV['PYTHONNOUSERSITE'] = '1'
SUBPROCESS_ENV['MPLBACKEND'] = 'Agg'


def run_stage(cmd, stdin_text=None, cwd=None):
    """Run a subprocess via the venv python with venv packages guaranteed first."""
    result = subprocess.run(
        cmd,
        input=stdin_text,
        text=True,
        capture_output=True,
        cwd=cwd or REPO_DIR,
        env=SUBPROCESS_ENV,
    )
    if result.returncode != 0:
        print('STDOUT:', result.stdout[-2000:])
        print('STDERR:', result.stderr[-2000:])
        raise RuntimeError(f'Command failed (exit {result.returncode}): {cmd}')


def load_mask_pngs(mask_dir, num_slices, H, W):
    """Load numbered PNG masks back into a (D, H, W) uint8 volume."""
    vol = np.zeros((num_slices, H, W), dtype=np.uint8)
    for i in range(num_slices):
        png_path = os.path.join(mask_dir, f'{i}.png')
        if os.path.exists(png_path):
            mask_img = cv2.imread(png_path, cv2.IMREAD_GRAYSCALE)
            if mask_img is not None:
                if mask_img.shape != (H, W):
                    mask_img = cv2.resize(mask_img, (W, H),
                                          interpolation=cv2.INTER_NEAREST)
                vol[i] = (mask_img > 127).astype(np.uint8)
    return vol


print('Helper functions defined.')

In [ ]:
import subprocess

for nii_path in nii_files:
    basename = os.path.basename(nii_path)
    stem     = basename.replace('_water.nii.gz', '')
    out_path = os.path.join(OUTPUT_DIR, f'{stem}_medclipsamv2.npz')

    if os.path.exists(out_path):
        print(f'Skipping (done): {stem}')
        continue

    print(f'\nProcessing: {stem}')
    img_sitk  = sitk.ReadImage(nii_path)
    img_array = sitk.GetArrayFromImage(img_sitk).astype(float)  # (D, H, W)
    D, H, W   = img_array.shape
    print(f'  Shape: {img_array.shape}')

    tmp_root  = tempfile.mkdtemp(prefix='mcs2a_')
    all_masks = {}

    try:
        # export slices once, reuse for all muscles
        png_dir = os.path.join(tmp_root, 'slices')
        export_slices_as_png(img_array, png_dir)

        for muscle_name, text_prompt in MUSCLES:
            print(f'  [{muscle_name}] prompt: "{text_prompt}"')

            sal_dir  = os.path.join(tmp_root, f'sal_{muscle_name}')
            post_dir = os.path.join(tmp_root, f'post_{muscle_name}')
            sam_dir  = os.path.join(tmp_root, f'sam_{muscle_name}')
            os.makedirs(sal_dir,  exist_ok=True)
            os.makedirs(post_dir, exist_ok=True)
            os.makedirs(sam_dir,  exist_ok=True)

            # Stage 1: BiomedCLIP saliency maps
            run_stage(
                [VENV_PY, SAL_SCRIPT,
                 '--input-path',  png_dir,
                 '--output-path', sal_dir,
                 '--val-path',    png_dir,
                 '--model-name',  'BiomedCLIP',
                 '--device',      DEVICE],
                stdin_text=text_prompt + '\n',
            )

            # Stage 2: postprocess saliency maps → coarse binary masks
            run_stage(
                [VENV_PY, POST_SCRIPT,
                 '--input-path',  png_dir,
                 '--output-path', post_dir,
                 '--sal-path',    sal_dir,
                 '--postprocess', 'kmeans',
                 '--filter'],
            )

            # Stage 3: SAM refinement
            run_stage(
                [VENV_PY, SAM_SCRIPT,
                 '--input',      png_dir,
                 '--mask-input', post_dir,
                 '--output',     sam_dir,
                 '--model-type', SAM_TYPE,
                 '--checkpoint', SAM_CKPT,
                 '--prompts',    'boxes',
                 '--device',     DEVICE],
            )

            # collect PNG masks → 3D volume
            all_masks[muscle_name] = load_mask_pngs(sam_dir, D, H, W)
            voxels = int(all_masks[muscle_name].sum())
            print(f'    -> {voxels:,} positive voxels')

        np.savez_compressed(out_path, **all_masks)
        print(f'  Saved -> {out_path}')

    except Exception as e:
        print(f'  ERROR on {stem}: {e}')

    finally:
        shutil.rmtree(tmp_root, ignore_errors=True)

print('\nAll done.')

In [ ]:
# sanity check
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.npz')))
print(f'Output files: {len(results)} / {len(nii_files)}')
if results:
    sample = np.load(results[0])
    print(f'  Sample: {results[0]}')
    for name in sorted(sample.files):
        arr = sample[name]
        print(f'    {name}: shape={arr.shape}  voxels={int(arr.sum()):,}')